# Reinforcement Learning Basics

In this demo, we will play around with the environments available through
[Gymnasium](https://gymnasium.farama.org/), the maintained successor of OpenAI Gym.
Everything below runs locally and headless -- no Colab, no virtual display.


## Libraries

We use PyTorch for the policy network and `imageio` to turn the frames the
environment gives us into a small animation we can look at afterwards.


In [ ]:
# Libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.distributions as distributions
import matplotlib.pyplot as plt
import numpy as np
import gymnasium as gym

import imageio


Check out the list of classic control tasks
[here](https://gymnasium.farama.org/environments/classic_control/). We'll be
experimenting with Cart Pole, where the object is to keep a pole upright on top of
a cart without it falling over.


In [ ]:
env = gym.make('CartPole-v1')

The observation space consits of four variables:

* Cart position (on a line)

* Cart velocity

* Pole angle

* Pole angular velocity


In [ ]:
print(env.observation_space)
print(env.observation_space.shape)

We can take two actions: either push the cart to the left or to the right.

In [ ]:
print(env.action_space)

The game is actually pretty hard! You can try your hand at it [here](https://jeffjar.me/cartpole.html). Now let's train a policy to beat it. We start by loading two different copies of the environment; one for training and one for testing.

In [ ]:
train_env = gym.make('CartPole-v1')
# The test environment also returns frames, so we can watch the trained policy play.
test_env = gym.make('CartPole-v1', render_mode='rgb_array')


We'll train a neural network that is actually incredibly simple: only two layers with dropout and ReLU.

In [ ]:
class PolicyNet(nn.Module):
  def __init__(self, input_dim, hidden_dim, output_dim, dropout=.5):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(input_dim, hidden_dim),
        nn.Dropout(dropout),
        nn.ReLU(),
        nn.Linear(hidden_dim, output_dim)
    )

  def forward(self, x):
    return self.net(x)

As we've done many times before, we initialize our architecture and optimizer. However, the loss will be slightly more tricky to compute than normal.

In [ ]:
input_dim = train_env.observation_space.shape[0]
print('input dim:', input_dim)

output_dim = train_env.action_space.n
print('output dim:', output_dim)

hidden_dim = 128

policy = PolicyNet(input_dim, hidden_dim, output_dim)
print(policy)

optimizer = optim.Adam(policy.parameters(), lr=0.01)

We will write a function for calculating a sequence of returns. With these in hand, we can calculate the empirical average over many trajectories as described in the lecture [notes](https://chinmayhegde.github.io/dl-notes/notes/lecture09/).

In [ ]:
def calculate_returns(rewards, discount_factor, normalize=True):
  returns = []
  R = 0
  for r in reversed(rewards):
    R = r + R * discount_factor
    returns.insert(0,R)

  returns = torch.tensor(returns)
  # Remember we normalize to differentiate between
  # "good" and "bad" actions
  if normalize:
    returns = (returns - returns.mean()) / returns.std()
  
  return returns

Once we have returns and log probabilities of the actions we take, we will compute the loss and optimize.

In [ ]:
def update_policy(returns, log_prob_actions, optimizer):
  returns = returns.detach()
  # Because we minimize (by convention) and we actually
  # want to maximize returns, we negate the expected sum
  loss = - (returns * log_prob_actions).sum()
  optimizer.zero_grad()

  loss.backward()
  optimizer.step()

  return loss.item()

Now we write the training and evaluation functions. Notice how training in RL is more involved than our standard applications.

In [ ]:
def train(env, policy, optimizer, discount_factor=.99):
  policy.train()

  log_prob_actions = []
  rewards = []
  done = False
  episode_reward = 0

  # Gymnasium's reset returns (observation, info)
  state, _ = env.reset()

  while not done:
    state = torch.FloatTensor(state).unsqueeze(0)
    action_pred = policy(state)
    action_prob = F.softmax(action_pred, dim = -1)

    dist = distributions.Categorical(action_prob)
    action = dist.sample() # we sample (rather than take the best)
    log_prob_action = dist.log_prob(action)

    # Gymnasium's step returns (obs, reward, terminated, truncated, info):
    # `terminated` means the pole fell over, `truncated` means we ran out of time.
    state, reward, terminated, truncated, _ = env.step(action.item())
    done = terminated or truncated

    log_prob_actions.append(log_prob_action)
    rewards.append(reward)

    episode_reward += reward

  log_prob_actions = torch.cat(log_prob_actions)
  returns = calculate_returns(rewards, discount_factor)
  loss = update_policy(returns, log_prob_actions, optimizer)

  return loss, episode_reward


In [ ]:
def evaluate(env, policy, return_frames=False):
  policy.eval()

  done = False
  episode_reward = 0
  state, _ = env.reset()
  frames = []

  while not done:
    if return_frames:
      # The environment was created with render_mode='rgb_array', so render()
      # hands back an (H, W, 3) uint8 image -- no display needed.
      frames += [env.render()]
    state = torch.FloatTensor(state).unsqueeze(0)

    action_pred = policy(state)
    action_prob = F.softmax(action_pred, dim=-1)

    action = torch.argmax(action_prob, dim=-1) # we take the best
    state, reward, terminated, truncated, _ = env.step(action.item())
    done = terminated or truncated
    episode_reward += reward

  if return_frames:
    return episode_reward, frames
  return episode_reward


Now we're ready to train. Notice the very jagged nature of the loss history. This is because RL is different from our regular training: our policy is stochastic and the reward is more "sparse".

In [ ]:
max_episodes = 500
num_trials = 25
reward_threshold = 475
print_every = 10

train_rewards = []
test_rewards = []

for episode in range(1, max_episodes+1):
  loss, train_reward = train(train_env, policy, optimizer)
  test_reward = evaluate(test_env, policy)

  train_rewards += [train_reward]
  test_rewards += [test_reward]

  mean_train_rewards = np.mean(train_rewards[-num_trials:])
  mean_test_rewards = np.mean(test_rewards[-num_trials:])

  if episode % print_every == 0:    
    print(f'| Episode: {episode:3} | Mean Train Rewards: {mean_train_rewards:5.1f} | Mean Test Rewards: {mean_test_rewards:5.1f} |')
    
  if mean_test_rewards >= reward_threshold:
    print(f'Reached reward threshold in {episode} episodes')
    break


In [ ]:
plt.figure(figsize=(8,8))
plt.plot(test_rewards, label='Test Reward')
plt.plot(train_rewards, label='Train Reward')
plt.xlabel('Episode', fontsize=20)
plt.ylabel('Reward', fontsize=20)
plt.hlines(reward_threshold, 0, len(test_rewards), color='r')
plt.legend(loc='lower right')
plt.grid()

In [ ]:
_, frames = evaluate(test_env, policy, return_frames=True)
print(f'The trained policy survived {len(frames)} steps.')


In [ ]:
# We have no display here, so instead of playing a video we lay a few of the
# frames out side by side: this is the trained cart keeping its pole upright.
picks = np.linspace(0, len(frames) - 1, 6).astype(int)

fig, axes = plt.subplots(1, len(picks), figsize=(18, 4))
for ax, k in zip(axes, picks):
    ax.imshow(frames[k])
    ax.set_title(f'step {k}')
    ax.axis('off')
plt.tight_layout()
plt.show()

# The whole episode, as an animated GIF next to this notebook.
imageio.mimsave('cartpole_trained.gif', frames[::2], fps=25, loop=0)
print('Wrote cartpole_trained.gif')
